# Entrega 2 - Pipeline de extração de features

Este notebook executa uma pipeline de extração de features para análise estatística a partir do dataset de memes.
As etapas incluem: YOLOv5 (detecção), FER (emoções faciais), estatísticas de cor, EasyOCR (OCR) e embeddings de texto (Sentence-Transformers).

Instruções rápidas:
1. Ative GPU no Colab (Runtime -> Change runtime type -> GPU).
2. Se for usar Google Drive monte-o na célula correspondente.
3. Ajuste o caminho das imagens / CSV de metadados conforme necessário.


In [ ]:
# 1) Instalação de dependências
# Execute esta célula no Colab. Em ambiente local pode pular algumas partes.
!pip install -q git+https://github.com/ultralytics/yolov5.git@master # yolov5
!pip install -q easyocr fer sentence-transformers opencv-python-headless pillow pandas pyarrow tqdm


In [ ]:
# 2) Verificar GPU e importar bibliotecas
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('Torch version:', torch.__version__)

import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import cv2
from PIL import Image

# easyocr wrapper (reaproveitando padrão do repositório)
import easyocr
_reader = None
def get_reader(model_storage_directory='/content/easyocr_models', gpu=True):
    global _reader
    if _reader is None:
        _reader = easyocr.Reader(['en'], gpu=gpu, download_enabled=True, model_storage_directory=model_storage_directory, verbose=False)
    return _reader

def run_easyocr_on_path(path, detail=1):

    reader = get_reader()
    return reader.readtext(path, detail=detail)


In [ ]:
# 3) Carregar modelos: YOLOv5 e Sentence-Transformers
import torch
# yolov5 via import
from yolov5 import load as yolo_load
model_yolo = yolo_load('yolov5s', pretrained=True)
model_yolo.conf = 0.25  # confidence threshold

# sentence-transformers para embeddings de texto
from sentence_transformers import SentenceTransformer
model_text = SentenceTransformer('all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')
print('Modelos carregados')


In [ ]:
# Funções utilitárias: deteção, crop, color stats, FER (opcional) e pipeline por imagem
from fer import FER
face_detector = FER(mtcnn=True)

def run_yolov5(image_path_or_array):
    # aceita path ou numpy array (BGR)
    if isinstance(image_path_or_array, (str,)):
        results = model_yolo(image_path_or_array)
        df = results.pandas().xyxy[0]
        img = cv2.imread(image_path_or_array)
    else:
        results = model_yolo(image_path_or_array)
        df = results.pandas().xyxy[0]
        img = image_path_or_array.copy()
    records = []
    for i, row in df.iterrows():
        x1, y1, x2, y2 = int(row.xmin), int(row.ymin), int(row.xmax), int(row.ymax)
        crop = img[y1:y2, x1:x2] if y2>y1 and x2>x1 else None
        records.append({
            'class': row.name,
            'label': row['name'],
            'conf': float(row.confidence),
            'bbox': (x1,y1,x2,y2),
            'crop': crop
        })
    return records

def color_stats_from_crop(crop):
    if crop is None:
        return {}
    # crop em BGR (OpenCV) -> converter para RGB
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    r,g,b = cv2.split(rgb)
    stats = {
        'r_mean': float(r.mean()), 'g_mean': float(g.mean()), 'b_mean': float(b.mean()),
        'r_std': float(r.std()), 'g_std': float(g.std()), 'b_std': float(b.std())
    }
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    h,s,v = cv2.split(hsv)
    stats.update({'h_mean': float(h.mean()), 's_mean': float(s.mean()), 'v_mean': float(v.mean())})
    return stats

def run_fer_on_crop(crop):
    # FER espera RGB PIL/numpy. Converter BGR->RGB
    if crop is None:
        return {}
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    # face_detector.detect_emotions aceita numpy RGB
    try:
        faces = face_detector.detect_emotions(rgb)
        if not faces:
            return {}
        # usar a primeira face encontrada dentro do crop
        top = faces[0]
        emotions = top.get('emotions', {})
        dominant = max(emotions.items(), key=lambda x: x[1])[0] if emotions else None
        return {'emotions': emotions, 'dominant_emotion': dominant}
    except Exception as e:
        return {'error': str(e)}

def run_easyocr_on_crop(crop):
    # EasyOCR readtext aceita path ou array RGB. Converter BGR->RGB e usar reader.readtext on array
    if crop is None:
        return []
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    results = get_reader().readtext(rgb, detail=1)
    return results

def text_to_embedding(text):
    if text is None or text.strip()=='' :
        return np.zeros((model_text.get_sentence_embedding_dimension(),), dtype=np.float32)
    emb = model_text.encode(text, convert_to_numpy=True, show_progress_bar=False)
    return emb.astype(np.float32)

def process_image(image_path):
    recs = []
    img = cv2.imread(image_path)
    detections = run_yolov5(image_path)
    # OCR global
    ocr_global = run_easyocr_on_path(image_path)
    ocr_global_text = ' '.join([r[1] for r in ocr_global]) if ocr_global else ''
    emb_global = text_to_embedding(ocr_global_text)
    for i, d in enumerate(detections):
        crop = d['crop']
        cstats = color_stats_from_crop(crop)
        fer_res = run_fer_on_crop(crop)
        ocr_crop = run_easyocr_on_crop(crop)
        ocr_text = ' '.join([r[1] for r in ocr_crop]) if ocr_crop else ''
        emb = text_to_embedding(ocr_text if ocr_text else ocr_global_text)
        record = {
            'image_path': image_path,
            'detection_id': i,
            'label': d['label'],
            'conf': d['conf'],
            'bbox': d['bbox'],
            'ocr_text': ocr_text,
            'ocr_global_text': ocr_global_text,
            'embedding': emb,
        }
        record.update(cstats)
        record.update(fer_res if isinstance(fer_res, dict) else {})
        recs.append(record)
    # se nenhuma detecção, ainda registrar linha com OCR global
    if not detections:
        recs.append({
            'image_path': image_path, 'detection_id': -1, 'label': None, 'conf': None, 'bbox': None,
            'ocr_text': '', 'ocr_global_text': ocr_global_text, 'embedding': emb_global
        })
    return recs


In [ ]:
# Execução em lote: apontar para pasta de imagens e processar algumas imagens de exemplo
IMAGES_DIR = '/content/images'  # ajuste se usar Drive
os.makedirs(IMAGES_DIR, exist_ok=True)
# Lista de imagens
import glob
images = glob.glob(f'{IMAGES_DIR}/**/*.jpg', recursive=True) + glob.glob(f'{IMAGES_DIR}/**/*.png', recursive=True)
print('Encontradas', len(images), 'imagens')
sample = images[:50]  # processar a té 50 por padrão
all_records = []
for p in tqdm(sample):
    try:
        recs = process_image(p  )
        all_records.extend(recs)
    except Exception as e:
        print('Erro em', p, e)

# Converter para DataFrame e salvar
if all_records:
    df = pd.DataFrame([{k:v for k,v in r.items() if k!='embedding'} for r in all_records])
    # salvar embeddings separadamente com alinhamento pelo index
    embs = np.stack([r['embedding'] for r in all_records])
    df.to_parquet('/content/feature_table.parquet', index=False)
    np.savez_compressed('/content/embeddings.npz', embeddings=embs)
    print('Salvo /content/feature_table.parquet e /content/embeddings.npz')
else:
    print('Nenhum registro gerado')


## Análise rápida
Carregue o Parquet e faça as análises estatísticas desejadas (contagens, distribuição de emoções, t-SNE nas embeddings, etc.).
